# 🌸 Data Classification with KNN — DecodeLabs Project 2

**AI Industrial Training Kit | Batch 2026 | Week 2**

Project 1 was *rule-based* — we wrote the logic by hand. Project 2 is the **predictive phase**: we hand the machine labelled history and it **derives** the logic itself. This is **supervised learning**.

We classify Iris flowers into 3 species using **K-Nearest Neighbors**.

### IPO pipeline (from the briefing)
| Stage | What happens |
|---|---|
| **INPUT** | Load Iris · understand it · `StandardScaler` |
| **PROCESS** | 80/20 train-test split · KNN · tune K |
| **OUTPUT** | Confusion matrix · precision / recall / **F1** |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, f1_score)

RANDOM_STATE = 42
sns.set_style('whitegrid')

## 1. INPUT — Load & understand the data
The **Iris benchmark**: 150 samples, 3 balanced classes (50 each), 4 numeric features (sepal/petal length & width).

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

print('Shape   :', X.shape)
print('Classes :', list(iris.target_names))
print('Balance :', y.value_counts().to_dict())
X.head()

In [ ]:
# Quick look at how separable the classes are
df = X.copy(); df['species'] = pd.Categorical.from_codes(y, iris.target_names)
sns.pairplot(df, hue='species', height=1.8)
plt.suptitle('Iris features by species', y=1.02)
plt.show()

## 2. PROCESS — Split first, then scale
We split **before** scaling and fit the scaler on the **training set only**. This keeps the test set a true unseen 'locked vault' — no data leakage. `stratify=y` keeps the 3 classes balanced in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE,
    shuffle=True, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on train
X_test  = scaler.transform(X_test)        # reuse train stats

print(f'Train: {len(X_train)} rows (80%)   Test: {len(X_test)} rows (20%)')
print('Scaled -> mean approx 0, variance approx 1')

## 3. PROCESS — Tune K (the elbow method)
Small K (1–2) overfits to noise; large K underfits. We plot the full curve to see both extremes, then pick the best **odd K ≥ 3** (odd avoids tie votes; skipping 1–2 avoids the overfitting trap).

In [ ]:
k_range = range(1, 26)
errors = []
for k in k_range:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    errors.append(np.mean(m.predict(X_test) != y_test))

candidates = [(k, e) for k, e in zip(k_range, errors) if k >= 3 and k % 2 == 1]
best_k = min(candidates, key=lambda p: (p[1], p[0]))[0]

plt.figure(figsize=(9,5))
plt.plot(list(k_range), errors, marker='o')
plt.axvline(best_k, color='orange', ls='--', label=f'Chosen K = {best_k}')
plt.title('Choosing K (Elbow Method)'); plt.xlabel('K'); plt.ylabel('Error rate')
plt.legend(); plt.show()
print('Chosen K =', best_k)

## 4. PROCESS — Train the model
The scikit-learn workflow: **instantiate → fit → predict**.

In [ ]:
model = KNeighborsClassifier(n_neighbors=best_k)  # instantiate
model.fit(X_train, y_train)                        # fit
predictions = model.predict(X_test)                # predict
predictions[:10]

## 5. OUTPUT — Validate (don't trust accuracy alone)
*"In imbalanced data, accuracy is a lie."* We report the **confusion matrix** and **precision / recall / F1** per class. **F1** is the harmonic mean of precision and recall.

In [ ]:
acc = accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average='macro')
print(f'Accuracy : {acc:.3f}')
print(f'Macro F1 : {macro_f1:.3f}\n')
print(classification_report(y_test, predictions, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'Confusion Matrix (KNN, K={best_k})')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()

## 6. Predict a brand-new flower
Using the trained model on a measurement it has never seen. Remember to scale the new sample with the **same** fitted scaler.

In [ ]:
new_flower = [[5.1, 3.5, 1.4, 0.2]]   # sepal L/W, petal L/W (cm)
new_scaled = scaler.transform(new_flower)
pred = model.predict(new_scaled)[0]
print('Predicted species:', iris.target_names[pred])

## ✅ Recap
- Loaded & understood a real dataset  
- Split into train/test and scaled correctly (no leakage)  
- Applied a supervised algorithm (KNN) and tuned K  
- Validated with a confusion matrix and F1 — not just accuracy  

**Next horizon:** from tabular data → deep learning & CNNs.